# Google Play Ingestion Monitoring Layer

This notebook adds a lightweight monitoring layer to the Phase 2 Google Play review-ingestion pipeline. It automatically summarizes the latest run, compares each app with its own recent history, validates the database, assigns `healthy`, `warning`, or `failing` status, and writes a clear run report.

The first thresholds are intentionally simple and transparent. They are based on the previous repeated-run and cadence results rather than universal assumptions. They can be adjusted after more production runs are available.


## 1. Import packages and set the monitoring configuration

The monitor uses the same controlled setup: 10 apps, 1,200 newest reviews per app, English, United States, and duplicate identity `source + app_id + review_id`.


In [1]:
import os
import json
import hashlib
import sqlite3
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

BASE_DIR = Path(os.environ.get("MONITORING_BASE_DIR", Path.cwd()))
WORK_DIR = BASE_DIR / "monitoring_workspace"
OUTPUT_DIR = BASE_DIR / "outputs"
REPORT_DIR = BASE_DIR / "reports"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_APP_COUNT = 10
EXPECTED_TARGET_PER_APP = 1200
REFERENCE_RUN_COUNT = 3

PACKAGE_NAME = (
    "phase2_cadence_runB_followup_github_upload_files_"
    "complete_report_20260714_170311_utc.zip"
)

print("Monitoring base folder:", BASE_DIR)
print("Reference runs per app:", REFERENCE_RUN_COUNT)
print("Expected apps:", EXPECTED_APP_COUNT)
print("Expected reviews per app:", EXPECTED_TARGET_PER_APP)


Monitoring base folder: /workspace/scratch/1b58880bdce1/monitoring_deliverable
Reference runs per app: 3
Expected apps: 10
Expected reviews per app: 1200


## 2. Load and verify the final Phase 2 package

The package manifest is checked before the database is opened. A missing non-database output is recorded as a warning signal. A missing, corrupted, or unloadable database is a failing signal.


In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def locate_source_package():
    explicit = os.environ.get("MONITORING_SOURCE_PACKAGE")
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates.extend([
        BASE_DIR / PACKAGE_NAME,
        Path("/content") / PACKAGE_NAME,
        Path.cwd() / PACKAGE_NAME,
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    try:
        from google.colab import files
        print("Upload the final Phase 2 complete-report ZIP package.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Please upload exactly one ZIP package.")
        uploaded_name = next(iter(uploaded))
        upload_path = BASE_DIR / uploaded_name
        upload_path.write_bytes(uploaded[uploaded_name])
        return upload_path.resolve()
    except ImportError as exc:
        raise FileNotFoundError(
            f"Could not find {PACKAGE_NAME}. Set MONITORING_SOURCE_PACKAGE."
        ) from exc


SOURCE_PACKAGE = locate_source_package()
if not zipfile.is_zipfile(SOURCE_PACKAGE):
    raise ValueError("The selected source package is not a valid ZIP file.")

EXTRACT_DIR = WORK_DIR / "source_package"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SOURCE_PACKAGE, "r") as archive:
    archive.extractall(EXTRACT_DIR)

manifest_path = EXTRACT_DIR / "final_package_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError("The source package manifest is missing.")

source_manifest_df = pd.read_csv(manifest_path)
manifest_checks = []

for row in source_manifest_df.itertuples(index=False):
    file_path = EXTRACT_DIR / row.file_name
    exists = file_path.exists()
    size_matches = exists and file_path.stat().st_size == int(row.size_bytes)
    hash_matches = exists and sha256_file(file_path) == str(row.sha256)
    manifest_checks.append({
        "file_name": row.file_name,
        "exists": bool(exists),
        "size_matches": bool(size_matches),
        "hash_matches": bool(hash_matches),
    })

source_manifest_validation_df = pd.DataFrame(manifest_checks)

database_zip_path = EXTRACT_DIR / "google_play_reviews_after_runB_followup.sqlite.zip"
if not database_zip_path.exists():
    raise FileNotFoundError("The required database archive is missing.")

DB_DIR = WORK_DIR / "database"
DB_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(database_zip_path, "r") as archive:
    archive.extractall(DB_DIR)

database_candidates = list(DB_DIR.rglob("*.sqlite"))
if len(database_candidates) != 1:
    raise FileNotFoundError(
        f"Expected one SQLite database but found {len(database_candidates)}."
    )

DB_PATH = database_candidates[0]
print("Source package:", SOURCE_PACKAGE.name)
print("Manifest files checked:", len(source_manifest_validation_df))
print("Database:", DB_PATH.name)
print("Database size:", f"{DB_PATH.stat().st_size / (1024 ** 2):.2f} MB")
display(source_manifest_validation_df.head())


,file_name,exists,size_matches,hash_matches
0,google_play_reviews_after_runB_followup.sqlite.zip,True,True,True
1,phase2_cadence_runA_runB_app_level_recommendations.csv,True,True,True
2,phase2_cadence_runA_runB_comparison_validation.csv,True,True,True
3,phase2_cadence_runA_runB_final_report.md,True,True,True
4,phase2_cadence_runA_runB_final_report_validation.csv,True,True,True


Source package: phase2_cadence_runB_followup_github_upload_files_complete_report_20260714_170311_utc.zip
Manifest files checked: 22
Database: google_play_reviews_after_runB_followup.sqlite
Database size: 84.68 MB


## 3. Read the latest run and its historical reference data

The latest completed Phase 2 run is the run being monitored. For behavior thresholds, the monitor uses the three most recent prior comparable runs with the same 10-app and 1,200-review configuration. The initial empty-database load is excluded because its 0% duplicate rate is not a steady-state baseline.


In [3]:
required_tables = {
    "phase2_reviews_raw",
    "phase2_reviews_cleaned",
    "phase2_apps",
    "phase2_ingestion_runs",
    "phase2_app_run_summary",
    "phase2_quality_flags",
}

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

existing_tables = set(pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table'",
    conn,
)["name"])

missing_tables = sorted(required_tables - existing_tables)
if missing_tables:
    raise ValueError(f"Required database tables are missing: {missing_tables}")

runs_df = pd.read_sql_query("""
    SELECT *
    FROM phase2_ingestion_runs
    ORDER BY run_started_at
""", conn)

app_runs_df = pd.read_sql_query("""
    SELECT
        s.*,
        r.run_label,
        r.frequency_label,
        r.run_started_at,
        r.status AS run_status
    FROM phase2_app_run_summary AS s
    JOIN phase2_ingestion_runs AS r
        ON s.run_id = r.run_id
    ORDER BY r.run_started_at, s.app_name
""", conn)

latest_run = runs_df.iloc[-1].copy()
LATEST_RUN_ID = str(latest_run["run_id"])
latest_apps_df = app_runs_df[
    app_runs_df["run_id"].eq(LATEST_RUN_ID)
].copy()

prior_reference_runs_df = runs_df[
    runs_df["run_started_at"].lt(latest_run["run_started_at"])
    & runs_df["status"].eq("completed")
    & runs_df["target_reviews_per_app"].eq(EXPECTED_TARGET_PER_APP)
    & runs_df["app_count"].eq(EXPECTED_APP_COUNT)
    & ~runs_df["frequency_label"].eq("once_daily_baseline")
].tail(REFERENCE_RUN_COUNT).copy()

reference_run_ids = prior_reference_runs_df["run_id"].tolist()
reference_apps_df = app_runs_df[
    app_runs_df["run_id"].isin(reference_run_ids)
].copy()

for frame in (latest_apps_df, reference_apps_df):
    frame["duplicate_rate"] = (
        frame["duplicates_skipped"]
        / frame["records_fetched"].replace(0, pd.NA)
    ).astype("Float64")
    frame["quality_flag_rate"] = (
        frame["quality_flag_count"]
        / frame["records_fetched"].replace(0, pd.NA)
    ).astype("Float64")

print("Latest run:", LATEST_RUN_ID)
print("Latest run status:", latest_run["status"])
print("Reference runs:", len(prior_reference_runs_df))
display(prior_reference_runs_df[[
    "run_label", "frequency_label", "run_started_at",
    "runtime_seconds", "new_records_inserted_total",
    "duplicates_skipped_total", "quality_flag_total",
]])


,run_label,frequency_label,run_started_at,runtime_seconds,new_records_inserted_total,duplicates_skipped_total,quality_flag_total
2,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,2026-07-09T04:04:39.642719+00:00,29.269241,5659,6341,12696
3,phase2_cadence_runA_twice_daily_test,twice_daily_same_day_second_run,2026-07-09T21:14:19.095514+00:00,28.142863,4395,7605,12721
4,phase2_cadence_runB_first_collection,runB_first_collection,2026-07-14T01:54:19.819646+00:00,30.761982,8638,3362,12630


Latest run: phase2_cadence_runB_followup_collection_20260714_163017
Latest run status: completed
Reference runs: 3


## 4. Run database and pipeline validation checks

Critical failures make the run `failing`. These checks cover run completion, app coverage, fetched/inserted/duplicate arithmetic, database row growth, raw-cleaned linkage, duplicate review identities, orphan quality flags, and foreign-key integrity.


In [4]:
def scalar(query, params=()):
    return pd.read_sql_query(query, conn, params=params).iloc[0, 0]


raw_rows = int(scalar("SELECT COUNT(*) FROM phase2_reviews_raw"))
cleaned_rows = int(scalar("SELECT COUNT(*) FROM phase2_reviews_cleaned"))
duplicate_identity_groups = int(scalar("""
    SELECT COUNT(*)
    FROM (
        SELECT source, app_id, review_id, COUNT(*) AS n
        FROM phase2_reviews_raw
        GROUP BY source, app_id, review_id
        HAVING COUNT(*) > 1
    )
"""))
raw_without_cleaned = int(scalar("""
    SELECT COUNT(*)
    FROM phase2_reviews_raw AS r
    LEFT JOIN phase2_reviews_cleaned AS c USING(review_key)
    WHERE c.review_key IS NULL
"""))
cleaned_without_raw = int(scalar("""
    SELECT COUNT(*)
    FROM phase2_reviews_cleaned AS c
    LEFT JOIN phase2_reviews_raw AS r USING(review_key)
    WHERE r.review_key IS NULL
"""))
orphan_quality_flags = int(scalar("""
    SELECT COUNT(*)
    FROM phase2_quality_flags AS q
    LEFT JOIN phase2_reviews_raw AS r USING(review_key)
    WHERE r.review_key IS NULL
"""))
foreign_key_violations = len(conn.execute("PRAGMA foreign_key_check").fetchall())

app_fetched_sum = int(latest_apps_df["records_fetched"].sum())
app_new_sum = int(latest_apps_df["new_records_inserted"].sum())
app_duplicate_sum = int(latest_apps_df["duplicates_skipped"].sum())
app_quality_sum = int(latest_apps_df["quality_flag_count"].sum())

validation_rows = [
    ("source_manifest_all_files_exist", source_manifest_validation_df["exists"].all(), "warning"),
    (
        "non_database_outputs_match_manifest",
        source_manifest_validation_df.loc[
            ~source_manifest_validation_df["file_name"].eq(
                "google_play_reviews_after_runB_followup.sqlite.zip"
            ),
            ["exists", "size_matches", "hash_matches"],
        ].all().all(),
        "warning",
    ),
    (
        "database_archive_matches_manifest",
        source_manifest_validation_df.loc[
            source_manifest_validation_df["file_name"].eq(
                "google_play_reviews_after_runB_followup.sqlite.zip"
            ),
            ["exists", "size_matches", "hash_matches"],
        ].all().all(),
        "failing",
    ),
    ("database_loaded", DB_PATH.exists(), "failing"),
    ("latest_run_completed", latest_run["status"] == "completed", "failing"),
    ("latest_run_has_finish_time", pd.notna(latest_run["run_finished_at"]), "failing"),
    ("expected_app_count", len(latest_apps_df) == EXPECTED_APP_COUNT, "failing"),
    ("expected_total_fetched", app_fetched_sum == EXPECTED_APP_COUNT * EXPECTED_TARGET_PER_APP, "warning"),
    ("app_fetched_matches_run", app_fetched_sum == int(latest_run["records_fetched_total"]), "failing"),
    ("app_new_inserts_match_run", app_new_sum == int(latest_run["new_records_inserted_total"]), "failing"),
    ("app_duplicates_match_run", app_duplicate_sum == int(latest_run["duplicates_skipped_total"]), "failing"),
    ("fetched_equals_new_plus_duplicates", app_fetched_sum == app_new_sum + app_duplicate_sum, "failing"),
    ("quality_flags_match_run", app_quality_sum == int(latest_run["quality_flag_total"]), "failing"),
    ("row_growth_matches_new_inserts", int(latest_run["review_rows_growth"]) == app_new_sum, "failing"),
    ("raw_rows_match_run_end", raw_rows == int(latest_run["review_rows_after"]), "failing"),
    ("raw_and_cleaned_counts_match", raw_rows == cleaned_rows, "failing"),
    ("no_duplicate_review_identities", duplicate_identity_groups == 0, "failing"),
    ("no_raw_rows_without_cleaned", raw_without_cleaned == 0, "failing"),
    ("no_cleaned_rows_without_raw", cleaned_without_raw == 0, "failing"),
    ("no_orphan_quality_flags", orphan_quality_flags == 0, "failing"),
    ("no_foreign_key_violations", foreign_key_violations == 0, "failing"),
    ("database_growth_nonnegative", float(latest_run["db_size_growth_mb"]) >= 0, "warning"),
]

validation_df = pd.DataFrame(
    validation_rows,
    columns=["validation_check", "passed", "severity_if_failed"],
)

print("Validation checks:", len(validation_df))
print("Failed checks:", int((~validation_df["passed"]).sum()))
display(validation_df)


,validation_check,passed,severity_if_failed
0,source_manifest_all_files_exist,True,warning
1,non_database_outputs_match_manifest,True,warning
2,database_archive_matches_manifest,True,failing
3,database_loaded,True,failing
4,latest_run_completed,True,failing
5,latest_run_has_finish_time,True,failing
6,expected_app_count,True,failing
7,expected_total_fetched,True,warning
8,app_fetched_matches_run,True,failing
9,app_new_inserts_match_run,True,failing


Validation checks: 22
Failed checks: 0


## 5. Define evidence-based initial thresholds

For each app, the baseline is the median of its three recent reference runs. Median absolute deviation (MAD) measures normal variation. A minimum practical margin prevents tiny historical differences from creating noisy alerts.

- Low new inserts: below `median − max(3×MAD, 50% of median, 25 records)`
- High duplicate rate: above `median + max(3×MAD, 10 percentage points)`
- Abnormal app runtime: above `median + max(3×MAD, 1 second)`
- Unexpected quality-flag change: more than `max(3×MAD, 0.10 flags per fetched record)` from the median

These are initial operational thresholds for this project, not universal Google Play rules.


In [5]:
def median_and_mad(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    median = float(values.median())
    mad = float((values - median).abs().median())
    return median, mad


threshold_rows = []

for app_id, current_row in latest_apps_df.set_index("app_id").iterrows():
    history = reference_apps_df[reference_apps_df["app_id"].eq(app_id)]
    new_median, new_mad = median_and_mad(history["new_records_inserted"])
    duplicate_median, duplicate_mad = median_and_mad(history["duplicate_rate"])
    runtime_median, runtime_mad = median_and_mad(history["runtime_seconds"])
    quality_median, quality_mad = median_and_mad(history["quality_flag_rate"])

    low_new_threshold = max(
        0.0,
        new_median - max(3 * new_mad, 0.50 * new_median, 25.0),
    )
    high_duplicate_threshold = min(
        1.0,
        duplicate_median + max(3 * duplicate_mad, 0.10),
    )
    high_runtime_threshold = runtime_median + max(3 * runtime_mad, 1.0)
    quality_change_threshold = max(3 * quality_mad, 0.10)

    threshold_rows.append({
        "app_name": current_row["app_name"],
        "app_id": app_id,
        "reference_run_count": len(history),
        "new_insert_median": new_median,
        "new_insert_mad": new_mad,
        "low_new_insert_warning_threshold": low_new_threshold,
        "duplicate_rate_median": duplicate_median,
        "duplicate_rate_mad": duplicate_mad,
        "high_duplicate_warning_threshold": high_duplicate_threshold,
        "runtime_median_seconds": runtime_median,
        "runtime_mad_seconds": runtime_mad,
        "high_runtime_warning_seconds": high_runtime_threshold,
        "quality_flag_rate_median": quality_median,
        "quality_flag_rate_mad": quality_mad,
        "quality_flag_rate_change_warning": quality_change_threshold,
    })

thresholds_df = pd.DataFrame(threshold_rows)
display(thresholds_df.round(4))


,app_name,app_id,reference_run_count,new_insert_median,new_insert_mad,low_new_insert_warning_threshold,duplicate_rate_median,duplicate_rate_mad,high_duplicate_warning_threshold,runtime_median_seconds,runtime_mad_seconds,high_runtime_warning_seconds,quality_flag_rate_median,quality_flag_rate_mad,quality_flag_rate_change_warning
0,DoorDash,com.dd.doordash,3,130.0,57.0,0.0,0.8917,0.0475,1.0000,0.7114,0.1413,1.7114,1.1108,0.0000,0.1
1,Duolingo,com.duolingo,3,6.0,1.0,0.0,0.9950,0.0008,1.0000,0.8753,0.0981,1.8753,1.0783,0.0000,0.1
2,Google Maps,com.google.android.apps.maps,3,231.0,47.0,90.0,0.8075,0.0392,0.9250,0.7970,0.1441,1.7970,0.8083,0.0050,0.1
3,Instagram,com.instagram.android,3,1199.0,1.0,599.5,0.0008,0.0008,0.1008,0.9213,0.0997,1.9213,1.3200,0.0050,0.1
4,Netflix,com.netflix.mediaclient,3,146.0,23.0,73.0,0.8783,0.0192,0.9783,0.7151,0.0547,1.7151,1.2958,0.0167,0.1
5,Reddit,com.reddit.frontpage,3,135.0,42.0,9.0,0.8875,0.0350,0.9925,0.6804,0.0089,1.6804,1.2108,0.0008,0.1
6,Spotify,com.spotify.music,3,723.0,143.0,294.0,0.3975,0.1192,0.7550,0.7442,0.0024,1.7442,1.0508,0.0025,0.1
7,TikTok,com.zhiliaoapp.musically,3,636.0,14.0,318.0,0.4700,0.0117,0.5700,0.8356,0.1524,1.8356,0.5408,0.0000,0.1
8,Uber,com.ubercab,3,402.0,85.0,147.0,0.6650,0.0708,0.8775,0.9020,0.0789,1.9020,1.1400,0.0058,0.1
9,YouTube,com.google.android.youtube,3,1199.0,1.0,599.5,0.0008,0.0008,0.1008,1.2362,0.0180,2.2362,1.0258,0.0042,0.1


## 6. Classify each app

An app is `failing` when collection failed or returned no records. It is `warning` when collection is partial or a metric crosses its app-specific threshold. Otherwise it is `healthy`.


In [6]:
app_health_rows = []

for _, current in latest_apps_df.iterrows():
    threshold = thresholds_df[
        thresholds_df["app_id"].eq(current["app_id"])
    ].iloc[0]

    failing_reasons = []
    warning_reasons = []

    error_message = current.get("error_message")
    if pd.notna(error_message) and str(error_message).strip():
        failing_reasons.append("app collection error")
    if int(current["records_fetched"]) == 0:
        failing_reasons.append("no records fetched")

    if 0 < int(current["records_fetched"]) < int(current["target_reviews"]):
        warning_reasons.append("partial fetched output")
    if float(current["new_records_inserted"]) < float(
        threshold["low_new_insert_warning_threshold"]
    ):
        warning_reasons.append("unusual drop in new inserts")
    if float(current["duplicate_rate"]) > float(
        threshold["high_duplicate_warning_threshold"]
    ):
        warning_reasons.append("unusually high duplicate rate")
    if float(current["runtime_seconds"]) > float(
        threshold["high_runtime_warning_seconds"]
    ):
        warning_reasons.append("abnormal app runtime")
    quality_change = abs(
        float(current["quality_flag_rate"])
        - float(threshold["quality_flag_rate_median"])
    )
    if quality_change > float(threshold["quality_flag_rate_change_warning"]):
        warning_reasons.append("unexpected quality-flag change")

    if failing_reasons:
        health_status = "failing"
        reasons = failing_reasons + warning_reasons
    elif warning_reasons:
        health_status = "warning"
        reasons = warning_reasons
    else:
        health_status = "healthy"
        reasons = ["all app-level checks within initial thresholds"]

    app_health_rows.append({
        "run_id": LATEST_RUN_ID,
        "app_name": current["app_name"],
        "app_id": current["app_id"],
        "health_status": health_status,
        "status_reason": "; ".join(reasons),
        "records_fetched": int(current["records_fetched"]),
        "new_records_inserted": int(current["new_records_inserted"]),
        "duplicates_skipped": int(current["duplicates_skipped"]),
        "duplicate_rate": float(current["duplicate_rate"]),
        "runtime_seconds": float(current["runtime_seconds"]),
        "quality_flag_count": int(current["quality_flag_count"]),
        "quality_flag_rate": float(current["quality_flag_rate"]),
        "low_new_insert_warning_threshold": float(threshold["low_new_insert_warning_threshold"]),
        "high_duplicate_warning_threshold": float(threshold["high_duplicate_warning_threshold"]),
        "high_runtime_warning_seconds": float(threshold["high_runtime_warning_seconds"]),
        "quality_flag_rate_change_warning": float(threshold["quality_flag_rate_change_warning"]),
        "error_message": "" if pd.isna(error_message) else str(error_message),
    })

app_health_df = pd.DataFrame(app_health_rows)
print(app_health_df["health_status"].value_counts().to_string())
display(app_health_df[[
    "app_name", "health_status", "status_reason", "records_fetched",
    "new_records_inserted", "duplicate_rate", "runtime_seconds",
    "quality_flag_count",
]])


,app_name,health_status,status_reason,records_fetched,new_records_inserted,duplicate_rate,runtime_seconds,quality_flag_count
0,DoorDash,healthy,all app-level checks within initial thresholds,1200,25,0.979167,0.927520,1332
1,Duolingo,healthy,all app-level checks within initial thresholds,1200,3,0.997500,0.972536,1296
2,Google Maps,healthy,all app-level checks within initial thresholds,1200,151,0.874167,0.842846,981
3,Instagram,healthy,all app-level checks within initial thresholds,1200,1196,0.003333,0.853803,1589
4,Netflix,healthy,all app-level checks within initial thresholds,1200,152,0.873333,0.856286,1491
5,Reddit,healthy,all app-level checks within initial thresholds,1200,61,0.949167,0.739151,1455
6,Spotify,healthy,all app-level checks within initial thresholds,1200,456,0.620000,0.770332,1274
7,TikTok,warning,unusually high duplicate rate,1200,406,0.661667,0.788907,616
8,Uber,healthy,all app-level checks within initial thresholds,1200,309,0.742500,0.783383,1348
9,YouTube,warning,unusually high duplicate rate,1200,994,0.171667,1.929823,1218


health_status
healthy    8
warning    2


## 7. Classify the overall run

The run inherits the most severe result. Any failed collection, database-loading problem, or failed critical validation makes it `failing`. App anomalies, incomplete expected outputs, abnormal runtime, or unusual metric changes make it `warning`.


In [7]:
reference_runtime_median, reference_runtime_mad = median_and_mad(
    prior_reference_runs_df["runtime_seconds"]
)
run_runtime_warning = (
    reference_runtime_median + max(3 * reference_runtime_mad, 15.0)
)

run_duplicate_rates = (
    prior_reference_runs_df["duplicates_skipped_total"]
    / prior_reference_runs_df["records_fetched_total"]
)
run_duplicate_median, run_duplicate_mad = median_and_mad(run_duplicate_rates)
run_duplicate_warning = min(
    1.0,
    run_duplicate_median + max(3 * run_duplicate_mad, 0.10),
)

run_new_median, run_new_mad = median_and_mad(
    prior_reference_runs_df["new_records_inserted_total"]
)
run_low_new_warning = max(
    0.0,
    run_new_median - max(3 * run_new_mad, 0.50 * run_new_median, 250.0),
)

run_quality_median, run_quality_mad = median_and_mad(
    prior_reference_runs_df["quality_flag_total"]
)
run_quality_change_warning = max(3 * run_quality_mad, 0.10 * run_quality_median)

current_duplicate_rate = (
    float(latest_run["duplicates_skipped_total"])
    / float(latest_run["records_fetched_total"])
)

run_warning_reasons = []
run_failing_reasons = []

failed_critical = validation_df[
    ~validation_df["passed"]
    & validation_df["severity_if_failed"].eq("failing")
]
failed_warning = validation_df[
    ~validation_df["passed"]
    & validation_df["severity_if_failed"].eq("warning")
]

if not failed_critical.empty:
    run_failing_reasons.append(
        "critical validation failed: "
        + ", ".join(failed_critical["validation_check"])
    )
if int(latest_run["errors_total"]) > 0:
    run_failing_reasons.append("one or more app collections failed")
if app_health_df["health_status"].eq("failing").any():
    run_failing_reasons.append("one or more apps classified as failing")

if not failed_warning.empty:
    run_warning_reasons.append(
        "warning validation failed: "
        + ", ".join(failed_warning["validation_check"])
    )
if app_health_df["health_status"].eq("warning").any():
    warning_apps = app_health_df.loc[
        app_health_df["health_status"].eq("warning"), "app_name"
    ].tolist()
    run_warning_reasons.append("app warnings: " + ", ".join(warning_apps))
if float(latest_run["runtime_seconds"]) > run_runtime_warning:
    run_warning_reasons.append("abnormal run runtime")
if current_duplicate_rate > run_duplicate_warning:
    run_warning_reasons.append("unusually high run duplicate rate")
if float(latest_run["new_records_inserted_total"]) < run_low_new_warning:
    run_warning_reasons.append("unusual run-level drop in new inserts")
if abs(float(latest_run["quality_flag_total"]) - run_quality_median) > run_quality_change_warning:
    run_warning_reasons.append("unexpected run-level quality-flag change")

if run_failing_reasons:
    RUN_HEALTH = "failing"
    RUN_REASONS = run_failing_reasons + run_warning_reasons
elif run_warning_reasons:
    RUN_HEALTH = "warning"
    RUN_REASONS = run_warning_reasons
else:
    RUN_HEALTH = "healthy"
    RUN_REASONS = ["all run-level and app-level checks passed"]

run_summary_df = pd.DataFrame([{
    "run_id": LATEST_RUN_ID,
    "run_label": latest_run["run_label"],
    "health_status": RUN_HEALTH,
    "status_reason": "; ".join(RUN_REASONS),
    "ingestion_status": latest_run["status"],
    "run_started_at": latest_run["run_started_at"],
    "run_finished_at": latest_run["run_finished_at"],
    "runtime_seconds": float(latest_run["runtime_seconds"]),
    "records_fetched_total": int(latest_run["records_fetched_total"]),
    "new_records_inserted_total": int(latest_run["new_records_inserted_total"]),
    "duplicates_skipped_total": int(latest_run["duplicates_skipped_total"]),
    "duplicate_rate": current_duplicate_rate,
    "errors_total": int(latest_run["errors_total"]),
    "database_row_growth": int(latest_run["review_rows_growth"]),
    "database_growth_mb": float(latest_run["db_size_growth_mb"]),
    "quality_flag_total": int(latest_run["quality_flag_total"]),
    "healthy_apps": int(app_health_df["health_status"].eq("healthy").sum()),
    "warning_apps": int(app_health_df["health_status"].eq("warning").sum()),
    "failing_apps": int(app_health_df["health_status"].eq("failing").sum()),
    "reference_run_count": len(prior_reference_runs_df),
    "run_runtime_warning_seconds": run_runtime_warning,
    "run_high_duplicate_warning_threshold": run_duplicate_warning,
    "run_low_new_insert_warning_threshold": run_low_new_warning,
    "run_quality_flag_change_warning": run_quality_change_warning,
}])

print("Overall monitoring status:", RUN_HEALTH.upper())
print("Reason:", "; ".join(RUN_REASONS))
display(run_summary_df.T)


,0
run_id,phase2_cadence_runB_followup_collection_20260714_163017
run_label,phase2_cadence_runB_followup_collection
health_status,warning
status_reason,"app warnings: TikTok, YouTube"
ingestion_status,completed
run_started_at,2026-07-14T16:34:49.444978+00:00
run_finished_at,2026-07-14T16:38:18.338912+00:00
runtime_seconds,30.016523
records_fetched_total,12000
new_records_inserted_total,3753


Overall monitoring status: WARNING
Reason: app warnings: TikTok, YouTube


## 8. Generate the automated monitoring outputs

The monitor writes a run summary, app-level health table, thresholds, validation checks, alerts, metadata, and a readable Markdown report. These files form the lightweight monitoring layer John requested.


In [8]:
generated_at = datetime.now(timezone.utc).isoformat()

alerts_df = app_health_df[
    ~app_health_df["health_status"].eq("healthy")
][[
    "run_id", "app_name", "app_id", "health_status", "status_reason"
]].copy()

run_alert = pd.DataFrame([{
    "run_id": LATEST_RUN_ID,
    "app_name": "ALL APPS",
    "app_id": "run_level",
    "health_status": RUN_HEALTH,
    "status_reason": "; ".join(RUN_REASONS),
}])

if RUN_HEALTH != "healthy":
    alerts_df = pd.concat([run_alert, alerts_df], ignore_index=True)

rule_definitions_df = pd.DataFrame([
    {
        "signal": "new inserts",
        "warning_rule": "current < median - max(3*MAD, 50% of median, 25)",
        "failing_rule": "none by itself; collection failure is evaluated separately",
        "reference": "three most recent prior comparable runs per app",
    },
    {
        "signal": "duplicate rate",
        "warning_rule": "current > median + max(3*MAD, 0.10)",
        "failing_rule": "none by itself",
        "reference": "three most recent prior comparable runs per app",
    },
    {
        "signal": "runtime",
        "warning_rule": "app > median + max(3*MAD, 1 sec); run > median + max(3*MAD, 15 sec)",
        "failing_rule": "none by itself",
        "reference": "three most recent prior comparable runs",
    },
    {
        "signal": "quality flags",
        "warning_rule": "absolute rate change > max(3*MAD, 0.10 flags/fetched record)",
        "failing_rule": "failed database quality validation",
        "reference": "three most recent prior comparable runs per app",
    },
    {
        "signal": "collection and database validation",
        "warning_rule": "partial fetch, missing non-database output, or negative DB file growth",
        "failing_rule": "collection error, database load/integrity failure, or failed critical validation",
        "reference": "fixed pipeline invariants",
    },
])

run_summary_path = OUTPUT_DIR / "monitoring_run_summary.csv"
app_health_path = OUTPUT_DIR / "monitoring_app_health.csv"
thresholds_path = OUTPUT_DIR / "monitoring_app_thresholds.csv"
rules_path = OUTPUT_DIR / "monitoring_rule_definitions.csv"
validation_path = OUTPUT_DIR / "monitoring_validation_checks.csv"
alerts_path = OUTPUT_DIR / "monitoring_alerts.csv"
manifest_validation_path = OUTPUT_DIR / "monitoring_source_manifest_validation.csv"
metadata_path = OUTPUT_DIR / "monitoring_metadata.json"
report_path = REPORT_DIR / "google_play_ingestion_monitoring_report.md"

run_summary_df.to_csv(run_summary_path, index=False)
app_health_df.to_csv(app_health_path, index=False)
thresholds_df.to_csv(thresholds_path, index=False)
rule_definitions_df.to_csv(rules_path, index=False)
validation_df.to_csv(validation_path, index=False)
alerts_df.to_csv(alerts_path, index=False)
source_manifest_validation_df.to_csv(manifest_validation_path, index=False)

metadata = {
    "generated_at": generated_at,
    "monitored_run_id": LATEST_RUN_ID,
    "health_status": RUN_HEALTH,
    "status_reasons": RUN_REASONS,
    "source_package": SOURCE_PACKAGE.name,
    "source_package_sha256": sha256_file(SOURCE_PACKAGE),
    "database_file": DB_PATH.name,
    "database_sha256": sha256_file(DB_PATH),
    "reference_run_ids": reference_run_ids,
    "reference_run_count": len(reference_run_ids),
    "threshold_method": "median and median absolute deviation with practical minimum margins",
    "expected_app_count": EXPECTED_APP_COUNT,
    "expected_target_per_app": EXPECTED_TARGET_PER_APP,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)


def markdown_table(frame, columns, formats=None):
    formats = formats or {}
    view = frame[columns].copy()
    for column, formatter in formats.items():
        view[column] = view[column].map(formatter)
    header = "| " + " | ".join(view.columns) + " |"
    separator = "|" + "|".join(["---"] * len(view.columns)) + "|"
    rows = [
        "| " + " | ".join(str(value) for value in row) + " |"
        for row in view.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator] + rows)


app_report_table = markdown_table(
    app_health_df,
    [
        "app_name", "health_status", "records_fetched",
        "new_records_inserted", "duplicates_skipped",
        "duplicate_rate", "runtime_seconds", "quality_flag_count",
        "status_reason",
    ],
    formats={
        "duplicate_rate": lambda value: f"{value:.2%}",
        "runtime_seconds": lambda value: f"{value:.2f}",
    },
)

failed_checks_text = "None"
failed_checks = validation_df[~validation_df["passed"]]
if not failed_checks.empty:
    failed_checks_text = ", ".join(failed_checks["validation_check"])

report_text = f"""# Google Play Ingestion Monitoring Report

## Overall status

- Monitoring status: **{RUN_HEALTH.upper()}**
- Monitored run: `{LATEST_RUN_ID}`
- Ingestion status: `{latest_run['status']}`
- Status reason: {'; '.join(RUN_REASONS)}
- Generated at: {generated_at}

## Run summary

| Signal | Value |
|---|---:|
| Runtime | {float(latest_run['runtime_seconds']):.2f} seconds |
| Total fetched | {int(latest_run['records_fetched_total']):,} |
| New inserts | {int(latest_run['new_records_inserted_total']):,} |
| Duplicates skipped | {int(latest_run['duplicates_skipped_total']):,} |
| Duplicate rate | {current_duplicate_rate:.2%} |
| App-level errors | {int(latest_run['errors_total']):,} |
| Database row growth | {int(latest_run['review_rows_growth']):,} |
| Database file growth | {float(latest_run['db_size_growth_mb']):.2f} MB |
| Quality flags | {int(latest_run['quality_flag_total']):,} |
| Healthy / warning / failing apps | {int(run_summary_df.iloc[0]['healthy_apps'])} / {int(run_summary_df.iloc[0]['warning_apps'])} / {int(run_summary_df.iloc[0]['failing_apps'])} |

## App-level health

{app_report_table}

## Validation result

- Checks run: {len(validation_df)}
- Failed checks: {int((~validation_df['passed']).sum())}
- Failed check names: {failed_checks_text}
- Duplicate review identity groups: {duplicate_identity_groups}
- Foreign-key violations: {foreign_key_violations}
- Raw review rows: {raw_rows:,}
- Cleaned review rows: {cleaned_rows:,}

## Threshold interpretation

The initial behavior thresholds use the three most recent prior comparable runs and are calculated separately for each app. Median and MAD are used because only a small historical sample is available and the apps have very different normal duplicate patterns.

A warning means the run completed but one or more signals should be reviewed. It does not automatically mean the pipeline failed. A failing status is reserved for collection failure, database loading or integrity failure, or a failed critical validation check.

Database growth in MB is reported only at the run level because SQLite file-page growth cannot be assigned reliably to individual apps.

These thresholds are project-specific starting points. They should be recalibrated after more routine production runs are collected.
"""

report_path.write_text(report_text, encoding="utf-8")

print("Automated monitoring outputs created:")
for path in sorted(list(OUTPUT_DIR.glob("monitoring_*")) + [report_path]):
    print("-", path.relative_to(BASE_DIR))


Automated monitoring outputs created:
- outputs/monitoring_alerts.csv
- outputs/monitoring_app_health.csv
- outputs/monitoring_app_thresholds.csv
- outputs/monitoring_metadata.json
- outputs/monitoring_output_manifest.csv
- outputs/monitoring_rule_definitions.csv
- outputs/monitoring_run_summary.csv
- outputs/monitoring_source_manifest_validation.csv
- outputs/monitoring_validation_checks.csv
- reports/google_play_ingestion_monitoring_report.md


## 9. Verify the monitoring deliverables

The final cell confirms that every required monitoring output exists, is nonempty, and can be traced with a SHA-256 hash.


In [9]:
required_output_paths = [
    run_summary_path,
    app_health_path,
    thresholds_path,
    rules_path,
    validation_path,
    alerts_path,
    manifest_validation_path,
    metadata_path,
    report_path,
]

output_manifest_rows = []
for path in required_output_paths:
    output_manifest_rows.append({
        "file_name": path.name,
        "relative_path": str(path.relative_to(BASE_DIR)),
        "exists": path.exists(),
        "nonempty": path.exists() and path.stat().st_size > 0,
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "sha256": sha256_file(path) if path.exists() else "",
    })

output_manifest_df = pd.DataFrame(output_manifest_rows)
output_manifest_path = OUTPUT_DIR / "monitoring_output_manifest.csv"
output_manifest_df.to_csv(output_manifest_path, index=False)

if not output_manifest_df["exists"].all():
    raise FileNotFoundError("One or more monitoring outputs are missing.")
if not output_manifest_df["nonempty"].all():
    raise ValueError("One or more monitoring outputs are empty.")

conn.close()

print("Monitoring layer completed successfully.")
print("Overall status:", RUN_HEALTH.upper())
print("Required outputs verified:", len(output_manifest_df))
display(output_manifest_df)
print("\nAutomated report:\n")
print(report_text)


,file_name,relative_path,exists,nonempty,size_bytes,sha256
0,monitoring_run_summary.csv,outputs/monitoring_run_summary.csv,True,True,811,1ba16fa640cd8457cbcb2a1d5368109c19599468d28cb7e9a1ea99c6f2b9f58a
1,monitoring_app_health.csv,outputs/monitoring_app_health.csv,True,True,2722,78bd7caf5fc6c99bd89e397bcc53b51f3572d75ab37954df084981711d0429df
2,monitoring_app_thresholds.csv,outputs/monitoring_app_thresholds.csv,True,True,2179,9d2f23134cc01ba2a36258eba15dc33b8263aa085dd61b85982ed9362db802e1
3,monitoring_rule_definitions.csv,outputs/monitoring_rule_definitions.csv,True,True,839,c943c822dd93ff7159b3d1647e12a0603f2e78b609036c66c71b1777d30a2dd5
4,monitoring_validation_checks.csv,outputs/monitoring_validation_checks.csv,True,True,919,4c08345686ee9eb6ec887f00d9a3bb128d20c0ddbf6c722911a50096f2281c40
5,monitoring_alerts.csv,outputs/monitoring_alerts.csv,True,True,421,9f74cc8afd9c2b39774b9453c33d127e86e450e59e3139432f139b7b3fdab67b
6,monitoring_source_manifest_validation.csv,outputs/monitoring_source_manifest_validation.csv,True,True,1490,198f8e6dc9201b7f294b44653d22b6f549a169203fff20a31605bb16f4e9020a
7,monitoring_metadata.json,outputs/monitoring_metadata.json,True,True,987,e0473e11fb47aec27991772cab388ecf4a78a4d09fcec54f2cafec4cfb7db840
8,google_play_ingestion_monitoring_report.md,reports/google_play_ingestion_monitoring_report.md,True,True,3002,6abece0d5521df202c311a0e5b7c563eaab2320faba17163152d07cc43332f38


Monitoring layer completed successfully.
Overall status: WARNING
Required outputs verified: 9

Automated report:

# Google Play Ingestion Monitoring Report

## Overall status

- Monitoring status: **WARNING**
- Monitored run: `phase2_cadence_runB_followup_collection_20260714_163017`
- Ingestion status: `completed`
- Status reason: app warnings: TikTok, YouTube
- Generated at: 2026-07-17T03:56:39.396231+00:00

## Run summary

| Signal | Value |
|---|---:|
| Runtime | 30.02 seconds |
| Total fetched | 12,000 |
| New inserts | 3,753 |
| Duplicates skipped | 8,247 |
| Duplicate rate | 68.73% |
| App-level errors | 0 |
| Database row growth | 3,753 |
| Database file growth | 10.59 MB |
| Quality flags | 12,600 |
| Healthy / warning / failing apps | 8 / 2 / 0 |

## App-level health

| app_name | health_status | records_fetched | new_records_inserted | duplicates_skipped | duplicate_rate | runtime_seconds | quality_flag_count | status_reason |
|---|---|---|---|---|---|---|---|---|
| DoorDash 